# Create the wave metamodel

Pipeline (all settings come from ``NorthCarolina.json`` → ``metamodel``):

1. **Train / load** MDA + PCA + GP models per goal (`fit_models`).
2. **Bulk reconstruction** — predict `hs / tp / wd / ds` at every output site
   for a chosen time window (`reconstruct_bulk`). Missing sea/swell timesteps
   are left as NaN.
3. **Zero-fill** missing forcing timesteps (`apply_missing_forcing_zeros`).
4. **Spectral reconstruction** — sum sea + swell partitions (`reconstruct_spectral`).
   Start with a small `stations` subset, then widen.
5. **Save** to `outputs/spectral_metamodel.nc`.
4. **Save** the result to `outputs/bulk_metamodel.nc`.

In [1]:
from utils import metamodel as mm
from utils.config import GOALS, METAMODEL, reload

reload()  # pick up edits to NorthCarolina.json without restarting kernel
print(f"goals            : {len(GOALS)}")
print(f"MDA points       : {METAMODEL['num_mda_points']}")
print(f"spectral output  : {mm.spectral_output_path()}")

goals            : 14
MDA points       : 500
spectral output  : /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc


## Parameters

Tweak these while experimenting — especially `time` and `stations` for the
spectral step.

In [7]:
# --- training --------------------------------------------------------------
force_retrain = False  # set True to overwrite existing .pkl models

# --- bulk reconstruction ---------------------------------------------------
time_bulk = "1986-01"  # any xarray time selector (month, day, slice, ...)

# --- spectral reconstruction (bottleneck) ------------------------------------
time_spec = "1986-01-15"  # None → use full bulk window; or e.g. "1986-01-15"
stations = "isobath"  # flexible: int, slice, list, "buoy", "isobath", name
# stations = slice(0, 3)   # quick smoke test on first 3 sites
# stations = -1              # single site by index

## 1 · Train / load models

In [3]:
models = mm.fit_models(force_retrain=force_retrain)
len(models["mda"]), len(models["pca"]), len(models["gp"])

(14, 14, 14)

## 2 · Bulk reconstruction

In [4]:
bulk = mm.reconstruct_bulk(time_bulk, models=models)
print(f"size : {bulk.nbytes * 1e-9:.2f} GB")
bulk

INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed max custom scaler for ds is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_u is bigger than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_u is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_v is bigger than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_v is lower than datapoint
INFO:ExactGPInterpolation:Subset data preprocessed successfully
INFO:ExactGPInterpolation:Predicting target variable: PC1
INFO:ExactGPInterpolation:Predicting target variable: PC2
INFO:ExactGPInterpolation:Predicting target variable: PC3
INFO:ExactGPInterpolation:Predicting target variable: PC4
INFO:ExactGPInterpolation:Predicting target variab

size : 0.36 GB


<xarray.Dataset> Size: 360MB
Dimensions:         (time: 744, goal: 14, partition: 5, stations: 216)
Coordinates:
  * time            (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
  * goal            (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude        (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude       (goal) float64 112B -75.5 -75.25 -75.06 ... -78.62 -79.0
    seapoint        (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06
  * partition       (partition) object 40B 'bulk' 'sea' ... 'swell2' 'swell3'
  * stations        (stations) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    station_source  (stations) <U7 6kB 'isobath' 'isobath' ... 'buoy' 'buoy'
    station_label   (stations) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels'
    station_depth   (stations) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan
Data variables:
    hs              (goal, partition, time, stations) float64 90MB 0.5362 ......
    tp              (goal, partition, time, stations) float64 90MB 4.373 ... ...
    wd              (goal, partition, time, stations) float64 90MB 175.5 ... ...
    ds              (goal, partition, time, stations) float64 90MB 37.0 ... 9.8

## 2b · Zero-fill missing forcing

Sea/swell partitions can be NaN when a component is absent. The metamodel only
predicts valid timesteps; this step sets those gaps to ``0.0`` before the
spectral sum.

In [5]:
bulk = mm.apply_missing_forcing_zeros(bulk)
bulk

<xarray.Dataset> Size: 360MB
Dimensions:         (time: 744, goal: 14, partition: 5, stations: 216)
Coordinates:
  * time            (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
  * goal            (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude        (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude       (goal) float64 112B -75.5 -75.25 -75.06 ... -78.62 -79.0
    seapoint        (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06
  * partition       (partition) object 40B 'bulk' 'sea' ... 'swell2' 'swell3'
  * stations        (stations) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    station_source  (stations) <U7 6kB 'isobath' 'isobath' ... 'buoy' 'buoy'
    station_label   (stations) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels'
    station_depth   (stations) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan
Data variables:
    hs              (goal, partition, time, stations) float64 90MB 0.5362 ......
    tp              (goal, partition, time, stations) float64 90MB 4.373 ... ...
    wd              (goal, partition, time, stations) float64 90MB 175.5 ... ...
    ds              (goal, partition, time, stations) float64 90MB 37.0 ... 9.8

## 3 · Spectral reconstruction

Start with a **small** `stations` selection above, then scale up.

In [8]:
spectral = mm.reconstruct_spectral(bulk, stations=stations, time=time_spec)
spectral

<xarray.Dataset> Size: 107kB
Dimensions:         (site: 185, time: 24)
Coordinates:
  * site            (site) <U6 4kB '5m_10' '5m_11' '5m_12' ... '15m_82' '15m_83'
    station_label   (site) <U10 7kB '5m' '5m' '5m' '5m' ... '15m' '15m' '15m'
    station_depth   (site) float64 1kB -5.0 -5.0 -5.0 -5.0 ... -15.0 -15.0 -15.0
    station         (site) <U6 4kB '5m_10' '5m_11' '5m_12' ... '15m_82' '15m_83'
  * time            (time) datetime64[ns] 192B 1986-01-15 ... 1986-01-15T23:0...
    station_source  <U7 28B 'isobath'
Data variables:
    hs              (site, time) float64 36kB 0.3163 0.3177 ... 1.202 1.148
    tp              (site, time) float32 18kB 7.189 7.27 6.795 ... 6.299 6.393
    dm              (site, time) float64 36kB 29.16 33.55 12.21 ... 79.68 80.9
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

## 4 · Save

In [9]:
out_path = mm.save_spectral(spectral, "outputs/spectral_recon_test.nc")
print(f"wrote {out_path}")

wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_recon_test.nc
